In [1]:
import warnings
import sys

if not sys.warnoptions:
    warnings.simplefilter("ignore")
import os
import glob
import numpy as np
from scipy import stats

from nilearn import datasets, surface
from nilearn import plotting
from nilearn.image import resample_to_img, index_img
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn.surface import SurfaceImage
import nibabel as nib

from brainiak import image, io
from brainiak.isc import isc, isfc, permutation_isc
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
 
from scipy import stats

sns.set(style="white", context="notebook", font_scale=1, rc={"lines.linewidth": 2})

In [2]:
from pathlib import Path
from tqdm import tqdm
 
from nilearn.maskers import NiftiMasker
from nilearn.image import resample_to_img, index_img
 
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
 
from scipy import stats

from scipy import stats
from joblib import Parallel, delayed
from sklearn.neighbors import KDTree

In [52]:
DESTRIEUX_ROIS = {
    # --- Auditory & speech ---
    "Heschl":            "G_temp_sup-G_T_transv",
    "STG_lateral":       "G_temp_sup-Lateral",
    "planum_temporale":  "G_temp_sup-Plan_tempo",
    "STS":               "S_temporal_sup",
    # --- Semantic / incongruity resolution ---
    "IFG_triangularis":  "G_front_inf-Triangul",
    "IFG_opercularis":   "G_front_inf-Opercular",
    "MTG":               "G_temporal_middle",
    # --- Social cognition / ToM ---
    "temporal_pole":     "G_temporal_poles",
    "angular_gyrus":     "G_pariet_inf-Angular",
    "supramarginal":     "G_pariet_inf-Supramar",
    # --- Reward & affect ---
    "ACC_anterior":      "G_and_S_cingul-Ant",
    "ACC_mid_ant":       "G_and_S_cingul-Mid-Ant",
    # --- Default mode ---
    "precuneus":         "G_precuneus",
    "parahippocampal":   "G_oc-temp_med-Parahip",
}


In [4]:
bids_dir = "/home/NEU480/datasets/narratives/"
fmriprep_dir = bids_dir + "derivatives/fmriprep/"
cleaned_dir = bids_dir + "derivatives/afni-nosmooth/"
stimuli_dir = bids_dir + "stimuli/"
transcript_dir =  "/home/vm5631/stimuli/transcripts_with_events/"
dir_nilearn = "/home/NEU480/datasets/nilearn_data"
code_dir = bids_dir + "code/"
net_id = os.environ['USER']
scratch_folder = f"/scratch/network/{net_id}/"
thesis_folder = scratch_folder + "thesis/"

In [5]:
# Load the fsaverage6 mesh once (needed to construct SurfaceImage)
fsaverage6 = datasets.fetch_surf_fsaverage('fsaverage6', data_dir=thesis_folder)
fsaverage6_mesh = surface.PolyMesh(
    left=fsaverage6['pial_left'],
    right=fsaverage6['pial_right']
)

[fetch_surf_fsaverage] Dataset found in /scratch/network/vm5631/thesis/fsaverage6

In [6]:
import json
import pandas as pd


def json_to_events(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    events_list = []
    for event in data['annotations']['events']:
        events_list.append({
            'onset': event['start'],
            'duration': event['end'] - event['start'],
            'trial_type': event['label']  # e.g., 'punchline' or 'audience_laughter'
        })
    
    return pd.DataFrame(events_list)

In [7]:
def fetch_shared_punchlines(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    events_list = []
    for event in data['annotations']['events']:
        if 'shared_punchline_id' in event:
            events_list.append({
                'onset': event['start'],
                'duration': event['end'] - event['start'],
                'trial_type': event['label']  # e.g., 'punchline' or 'audience_laughter'
            })
    
    return pd.DataFrame(events_list)

In [8]:
events_live = json_to_events('stimuli/transcripts_with_events/pieman_audio.json')
events_live

,onset,duration,trial_type
0,69.218,2.582,punchline
1,71.954,1.546,audience_laughter
2,79.249,0.740,punchline
3,80.244,1.407,audience_laughter
4,86.093,2.561,punchline
5,88.592,2.128,audience_laughter
6,95.318,0.680,punchline
7,96.185,1.531,audience_laughter
8,115.150,1.421,punchline
9,131.451,0.861,punchline


In [9]:
events_pni = json_to_events('stimuli/transcripts_with_events/piemanpni_audio.json')
events_pni['onset'] = events_pni['onset'] + 12

In [10]:
events_pni

,onset,duration,trial_type
0,78.491,3.963,punchline
1,88.179,1.421,punchline
2,93.243,2.722,punchline
3,100.343,1.241,punchline
4,116.676,1.921,punchline
5,133.258,1.000,punchline
6,135.198,1.721,punchline
7,141.760,2.961,punchline
8,154.424,2.481,punchline
9,163.979,2.782,punchline


In [11]:
punchlines_live = events_live[events_live['trial_type'] == 'punchline'].reset_index(drop=True)
punchlines_live

,onset,duration,trial_type
0,69.218,2.582,punchline
1,79.249,0.740,punchline
2,86.093,2.561,punchline
3,95.318,0.680,punchline
4,115.150,1.421,punchline
5,131.451,0.861,punchline
6,135.898,2.345,punchline
7,143.037,2.682,punchline
8,156.448,1.901,punchline
9,164.053,1.902,punchline


In [12]:
punchlines_pni = events_pni[events_pni['trial_type'] == 'punchline'].reset_index(drop=True)
punchlines_pni

,onset,duration,trial_type
0,78.491,3.963,punchline
1,88.179,1.421,punchline
2,93.243,2.722,punchline
3,100.343,1.241,punchline
4,116.676,1.921,punchline
5,133.258,1.000,punchline
6,135.198,1.721,punchline
7,141.760,2.961,punchline
8,154.424,2.481,punchline
9,163.979,2.782,punchline


In [13]:
shared_punchlines_live = fetch_shared_punchlines('stimuli/transcripts_with_events/pieman_audio.json')
shared_punchlines_live

,onset,duration,trial_type
0,69.218,2.582,punchline
1,79.249,0.740,punchline
2,86.093,2.561,punchline
3,115.150,1.421,punchline
4,131.451,0.861,punchline
5,135.898,2.345,punchline
6,143.037,2.682,punchline
7,156.448,1.901,punchline
8,164.053,1.902,punchline
9,169.812,2.442,punchline


In [14]:
shared_punchlines_pni = fetch_shared_punchlines('stimuli/transcripts_with_events/piemanpni_audio.json')
shared_punchlines_pni['onset'] = shared_punchlines_pni['onset'] + 12

In [15]:
shared_punchlines_pni

,onset,duration,trial_type
0,78.491,3.963,punchline
1,88.179,1.421,punchline
2,93.243,2.722,punchline
3,116.676,1.921,punchline
4,133.258,1.000,punchline
5,135.198,1.721,punchline
6,141.760,2.961,punchline
7,154.424,2.481,punchline
8,163.979,2.782,punchline
9,169.142,2.861,punchline


In [16]:
tr = 1.5
n_scans_live = 300
n_scans_pni = 294
trim_front_live = 10
trim_back_live = 8
trim_front_pni = 14
trim_back_pni = 13
exclude_subjects_live = [1, 13, 14, 21, 22, 38, 56, 68, 69]
subs_live = [(i + 1) for i in range(16, 82)]
exclude_subjects_pni = [266, 268, 269, 270, 271, 280]
subs_pni = [(i + 1) for i in range(264, 315)]
frame_times_live = np.arange(n_scans_live) * tr
frame_times_pni = np.arange(n_scans_pni) * tr

In [17]:
def remove_subjects(sub_list, exclude_list):
    cleaned_list = sub_list
    for sub in exclude_list:
        if sub in sub_list:
            cleaned_list.remove(sub) 
    return cleaned_list

In [18]:
subs_live = remove_subjects(subs_live, exclude_subjects_live)

In [19]:
subs_pni = remove_subjects(subs_pni, exclude_subjects_pni)
subs_pni.append(127)

In [20]:
import random

def get_random_items(data_list, count):
    # random.sample raises an error if count > len(data_list)
    if count > len(data_list):
        return "Count is larger than the list size!"
    
    return random.sample(data_list, count)

In [47]:
live_random = get_random_items(subs_live, 45)
live_random

[25,
 44,
 30,
 33,
 76,
 72,
 80,
 41,
 23,
 62,
 28,
 52,
 54,
 70,
 55,
 26,
 47,
 73,
 36,
 75,
 32,
 59,
 50,
 49,
 43,
 77,
 78,
 34,
 61,
 71,
 46,
 64,
 60,
 58,
 51,
 81,
 42,
 48,
 74,
 17,
 40,
 82,
 63,
 39,
 57]

In [48]:
pni_random = get_random_items(subs_pni, 45)
pni_random

[299,
 292,
 298,
 127,
 288,
 307,
 314,
 284,
 286,
 265,
 315,
 312,
 297,
 281,
 285,
 306,
 302,
 276,
 295,
 305,
 282,
 290,
 278,
 313,
 273,
 274,
 301,
 309,
 310,
 308,
 296,
 294,
 289,
 287,
 303,
 279,
 272,
 277,
 291,
 300,
 311,
 275,
 293,
 304,
 283]

In [23]:
def get_data(subs, task):
    data = []
    for sub in subs:
        gifti_imgs = {}
        for hemi, side in [('L', 'lh'), ('R', 'rh')]:
            file_name = (
                f"sub-{sub:03d}_task-{task}_space-fsaverage6"
                f"_hemi-{hemi}_desc-clean.func.gii"
            )
            file_dir = f"{cleaned_dir}sub-{sub:03d}/func/"
            gifti_imgs[side + "_func"] = file_dir + file_name  # pass path, not loaded image
        gifti_imgs['confounds'] = None
        gifti_imgs['id'] = sub
        data.append(gifti_imgs)
    return data

In [49]:
LIVE_PARTICIPANTS = get_data(live_random, "pieman")

In [50]:
STUDIO_PARTICIPANTS = get_data(pni_random, "piemanpni")

In [26]:
# Number of permutations for the group-level permutation test
N_PERMUTATIONS = 5000
 
OUTPUT_DIR = Path("results/mvpa")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [27]:
"""
Surface MVPA Pipeline: Punchline vs Non-Punchline Classification
================================================================
Surface-based version of the MVPA pipeline. Uses GIFTI .func.gii
functional data, extracts per-trial LSS beta patterns within
Destrieux parcels on fsaverage5, classifies punchline vs baseline
with a linear SVM (LOO-CV), and compares accuracy between the live
and studio groups using a permutation test.

Key differences from the volumetric pipeline
---------------------------------------------
- Functional data is GIFTI (.func.gii), not NIfTI
- ROIs come from the Destrieux surface atlas (nilearn built-in,
  already on fsaverage5 — no volumetric mask resampling needed)
- GLM is run directly on vertex timeseries arrays using
  make_first_level_design_matrix + run_glm, bypassing
  FirstLevelModel (which is NIfTI-only)
- Analysis is run separately per hemisphere; results are merged
  at the group-comparison stage

Requirements: nilearn, nibabel, scikit-learn, scipy, numpy, pandas, joblib
"""
HRF_MODEL   = "spm"
N_PERMUTATIONS = 5000
N_JOBS         = -1   # -1 = all cores; set to e.g. 4 on a shared cluster

OUTPUT_DIR = Path("results/mvpa")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [44]:
def load_destrieux_atlas() -> dict:
    """
    Fetch the Destrieux atlas on fsaverage5 from nilearn, then resample
    labels to fsaverage6 via nearest-neighbour on the registration sphere.
    """
    from scipy.spatial import cKDTree
    from nilearn import surface

    # --- Fetch atlas (fsaverage5) and both sphere meshes ---
    destrieux = datasets.fetch_atlas_surf_destrieux(data_dir=thesis_folder,
                                                     verbose=0)
    fsavg5 = datasets.fetch_surf_fsaverage("fsaverage5", data_dir=thesis_folder)
    fsavg6 = datasets.fetch_surf_fsaverage("fsaverage6", data_dir=thesis_folder)

    # --- Resample helper: nearest-neighbour on the sphere ---
    def _resample(sphere_src, sphere_tgt, labels_src):
        coords_src = surface.load_surf_mesh(sphere_src).coordinates
        coords_tgt = surface.load_surf_mesh(sphere_tgt).coordinates
        tree = cKDTree(coords_src)
        _, idx = tree.query(coords_tgt)
        return labels_src[idx]

    atlas = {}
    sphere_keys = {
        "lh": ("sphere_left",  "map_left"),
        "rh": ("sphere_right", "map_right"),
    }
    for hemi, (sph_key, map_key) in sphere_keys.items():
        labels_fs5 = np.array(destrieux[map_key])
        labels_fs6 = _resample(fsavg5[sph_key], fsavg6[sph_key], labels_fs5)
        atlas[hemi] = {
            "labels": list(destrieux.labels),
            "map":    labels_fs6,
        }

    # Decode byte-strings to str
    for hemi in ("lh", "rh"):
        atlas[hemi]["labels"] = [
            lbl.decode() if isinstance(lbl, bytes) else lbl
            for lbl in atlas[hemi]["labels"]
        ]
    return atlas

def get_parcel_mask(atlas_hemi: dict, parcel_name: str) -> np.ndarray:
    """
    Return a boolean vertex mask for the named Destrieux parcel.
    Raises ValueError if the parcel name is not found.
    """
    labels = atlas_hemi["labels"]
    pmap   = atlas_hemi["map"]

    # Label indices are 1-based in the Destrieux atlas (0 = background)
    matches = [i for i, lbl in enumerate(labels) if parcel_name in lbl]
    if not matches:
        available = "\n  ".join(labels)
        raise ValueError(
            f"Parcel '{parcel_name}' not found in Destrieux atlas.\n"
            f"Available labels:\n  {available}"
        )
    # There should be exactly one match per parcel per hemisphere
    parcel_idx = matches[0]
    return pmap == parcel_idx

In [29]:
# =============================================================================
# 3. LOAD GIFTI FUNCTIONAL DATA AND BASELINE SAMPLING
# =============================================================================

def load_gifti_timeseries(gifti_path: str) -> np.ndarray:
    """
    Load a .func.gii file and return a (n_vertices, n_timepoints) array.
    Each darrays entry in the GIFTI corresponds to one timepoint.
    """
    img = nib.load(gifti_path)
    # Stack along time axis: each darray is shape (n_vertices,)
    data = np.column_stack([da.data for da in img.darrays])  # (n_vertices, n_trs)
    return data


def load_confounds(tsv_path: str | None,
                   columns: list | None) -> np.ndarray | None:
    """Returns (n_trs, n_confounds) array or None."""
    if tsv_path is None or columns is None:
        return None
    df = pd.read_csv(tsv_path, sep="\t")
    available = [c for c in columns if c in df.columns]
    return df[available].fillna(0).values

def sample_baseline_onsets(punchline_df: pd.DataFrame,
                            func_duration: float,
                            tr: float,
                            n_samples: int,
                            min_gap_punchline: float = 12.0,
                            min_gap_self: float = 4.0,
                            run_edge_buffer: float = 20.0) -> np.ndarray:
    """
    Sample baseline onsets well-separated from punchlines and each other.
    See volumetric pipeline for full parameter documentation.
    """
    punchline_onsets = punchline_df["onset"].values
    candidates = np.arange(run_edge_buffer, func_duration - run_edge_buffer, tr)

    far = np.array([
        t for t in candidates
        if np.all(np.abs(t - punchline_onsets) > min_gap_punchline)
    ])

    if len(far) < n_samples:
        raise ValueError(
            f"Not enough baseline candidates: found {len(far)}, need {n_samples}. "
            f"Try reducing min_gap_punchline or run_edge_buffer."
        )

    rng     = np.random.default_rng(42)
    shuffled = rng.permutation(far)
    chosen  = []
    for t in shuffled:
        if not chosen or np.all(np.abs(t - np.array(chosen)) >= min_gap_self):
            chosen.append(t)
        if len(chosen) == n_samples:
            break

    if len(chosen) < n_samples:
        raise ValueError(
            f"Only found {len(chosen)} baseline onsets with min_gap_self={min_gap_self}s. "
            f"Try reducing min_gap_self."
        )
    return np.sort(chosen)

In [1]:
# =============================================================================
# 4. LSS ON SURFACE TIMESERIES
# =============================================================================

def build_lss_design_matrix(target_event: pd.DataFrame,
                              other_events: pd.DataFrame,
                              n_scans: int,
                              tr: float) -> pd.DataFrame:
    """
    Build a single-trial LSS design matrix with:
      - 'target'  : the trial of interest
      - 'others'  : all other trials lumped
    """
    lss_events = pd.concat([
        target_event.assign(trial_type="target"),
        other_events.assign(trial_type="others"),
    ], ignore_index=True)

    frame_times = np.arange(n_scans) * tr

    dm = make_first_level_design_matrix(
        frame_times=frame_times,
        events=lss_events,
        hrf_model=HRF_MODEL,
        drift_model=None,
        high_pass=0,
    )
    return dm


def lss_single_trial_beta(surface_data: np.ndarray,
                           target_event: pd.DataFrame,
                           other_events: pd.DataFrame,
                           confounds: np.ndarray | None,
                           tr: float) -> np.ndarray:
    """
    Fit one LSS GLM for a single trial on surface vertex timeseries.

    Parameters
    ----------
    surface_data  : (n_vertices, n_trs) array
    target_event  : single-row DataFrame with onset, duration, trial_type
    other_events  : remaining events DataFrame
    confounds     : (n_trs, n_confounds) array or None
    tr            : repetition time in seconds

    Returns
    -------
    beta_vector : (n_vertices,) beta estimates for the target regressor
    """
    n_vertices, n_scans = surface_data.shape
    dm = build_lss_design_matrix(target_event, other_events, n_scans, tr)

    # Append confound columns to design matrix if provided
    if confounds is not None:
        confound_df = pd.DataFrame(
            confounds,
            columns=[f"conf_{i}" for i in range(confounds.shape[1])]
        )
        dm = pd.concat([dm, confound_df.reset_index(drop=True)], axis=1)

    # run_glm expects (n_timepoints, n_vertices) — transpose surface_data
    X = dm.values                          # (n_scans, n_regressors)
    Y = surface_data.T                     # (n_scans, n_vertices)

    # OLS via normal equations — fast, no AR1 needed for LSS
    betas, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)  # (n_regressors, n_vertices)

    # Find the index of the 'target' regressor
    target_col = dm.columns.get_loc("target")
    return betas[target_col]  # (n_vertices,)


def lss_beta_series(surface_data: np.ndarray,
                    events_df: pd.DataFrame,
                    confounds: np.ndarray | None,
                    tr: float,
                    desc: str = "LSS") -> np.ndarray:
    """
    Run LSS for all trials in events_df.

    Returns
    -------
    beta_series : (n_trials, n_vertices) array
    """
    n_trials = len(events_df)

    def _one_trial(i):
        target = events_df.iloc[[i]].copy()
        others = events_df.drop(index=i).reset_index(drop=True)
        return lss_single_trial_beta(surface_data, target, others, confounds, tr)

    betas = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(_one_trial)(i)
        for i in tqdm(range(n_trials), desc=f"    {desc}", leave=False)
    )
    return np.vstack(betas)  # (n_trials, n_vertices)

NameError: name 'pd' is not defined

In [31]:
# =============================================================================
# 6. ROI EXTRACTION AND CLASSIFICATION
# =============================================================================

def classify_loo(X_punchline: np.ndarray,
                 X_baseline: np.ndarray) -> float:
    """
    Linear SVM LOO-CV: punchline (1) vs baseline (0).
    Returns mean accuracy (0–1).
    """
    X = np.vstack([X_punchline, X_baseline])
    y = np.array([1] * len(X_punchline) + [0] * len(X_baseline))

    clf = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1))
    loo = LeaveOneOut()

    correct = [
        clf.fit(X[train], y[train]).predict(X[test])[0] == y[test][0]
        for train, test in loo.split(X)
    ]
    return float(np.mean(correct))

In [32]:
# =============================================================================
# 7. PER-PARTICIPANT MVPA (ONE HEMISPHERE)
# =============================================================================

def run_hemi_mvpa(surface_data: np.ndarray,
                  atlas_hemi: dict,
                  events_df: pd.DataFrame,
                  confounds: np.ndarray | None,
                  tr: float,
                  hemi_label: str) -> dict:
    """
    Run the full MVPA pipeline for one hemisphere of one participant.

    Returns dict of {roi_name: accuracy} for all DESTRIEUX_ROIS.
    """
    n_trs        = surface_data.shape[1]
    run_duration = n_trs * tr
    median_dur   = float(events_df["duration"].median())

    # --- LSS betas: punchline trials ---
    print(f"    [{hemi_label}] LSS — punchline trials...")
    punch_betas = lss_beta_series(
        surface_data, events_df, confounds, tr, desc=f"{hemi_label} punchline"
    )  # (n_punchline, n_vertices)

    # --- LSS betas: matched baseline trials ---
    baseline_onsets = sample_baseline_onsets(
        events_df, run_duration, tr, n_samples=len(events_df)
    )
    baseline_events = pd.DataFrame({
        "onset":      baseline_onsets,
        "duration":   median_dur,
        "trial_type": "punchline",
    })

    print(f"    [{hemi_label}] LSS — baseline trials...")
    base_betas = lss_beta_series(
        surface_data, baseline_events, confounds, tr, desc=f"{hemi_label} baseline"
    )  # (n_baseline, n_vertices)

    # --- Classify within each ROI ---
    roi_accuracies = {}
    for roi_name, destrieux_label in DESTRIEUX_ROIS.items():
        try:
            parcel_mask = get_parcel_mask(atlas_hemi, destrieux_label)
        except ValueError as e:
            print(f"    WARNING: {e}")
            roi_accuracies[roi_name] = np.nan
            continue

        n_voxels = parcel_mask.sum()
        if n_voxels == 0:
            print(f"    WARNING: {roi_name} ({destrieux_label}) is empty in {hemi_label}, skipping.")
            roi_accuracies[roi_name] = np.nan
            continue

        X_punch = punch_betas[:, parcel_mask]  # (n_trials, n_parcel_vertices)
        X_base  = base_betas[:,  parcel_mask]

        acc = classify_loo(X_punch, X_base)
        roi_accuracies[roi_name] = acc

    return roi_accuracies

def run_participant_mvpa(participant: dict,
                          events_df: pd.DataFrame,
                          tr: float,
                          atlas: dict) -> dict:
    """
    Run MVPA for both hemispheres of one participant.
    Returns dict keyed by '{roi_name}_lh' and '{roi_name}_rh'.
    """
    confounds = None
    results   = {}

    for hemi, func_key in [("lh", "lh_func"), ("rh", "rh_func")]:
        surface_data = load_gifti_timeseries(participant[func_key])
        # surface_data: (n_vertices, n_trs)

        hemi_accs = run_hemi_mvpa(
            surface_data, atlas[hemi], events_df, confounds, tr, hemi_label=hemi
        )
        for roi_name, acc in hemi_accs.items():
            results[f"{roi_name}_{hemi}"] = acc

    return results

In [33]:
# =============================================================================
# 8. RUN ALL PARTICIPANTS IN A GROUP
# =============================================================================

def _run_one_participant(p: dict,
                          events_df: pd.DataFrame,
                          tr: float,
                          atlas: dict,
                          idx: int,
                          total: int) -> dict:
    label = p.get("lh_func", f"participant_{idx}")
    print(f"\n  [{idx+1}/{total}] {Path(label).name}")
    accs = run_participant_mvpa(p, events_df, tr, atlas)
    accs["participant"] = Path(label).name
    return accs


def run_group_mvpa(participants: list,
                   events_df: pd.DataFrame,
                   tr: float,
                   group_label: str,
                   atlas: dict) -> pd.DataFrame:
    print(f"\n{'='*60}")
    print(f"Group: {group_label}  |  {len(participants)} participants  |  TR={tr}s")
    print(f"Punchline events: {len(events_df)}")
    print(f"{'='*60}")

    rows = Parallel(n_jobs=N_JOBS, prefer="processes")(
        delayed(_run_one_participant)(p, events_df, tr, atlas, i, len(participants))
        for i, p in enumerate(participants)
    )

    df = pd.DataFrame(rows)
    df["group"] = group_label
    return df

In [34]:
# =============================================================================
# 9. GROUP-LEVEL PERMUTATION TEST
# =============================================================================

def permutation_ttest(a: np.ndarray,
                      b: np.ndarray,
                      n_permutations: int = 5000) -> tuple[float, float]:
    observed_t, _ = stats.ttest_ind(a, b, equal_var=False)
    combined = np.concatenate([a, b])
    n_a      = len(a)
    rng      = np.random.default_rng(0)

    null_t = np.array([
        stats.ttest_ind(
            (perm := rng.permutation(combined))[:n_a],
            perm[n_a:],
            equal_var=False
        )[0]
        for _ in range(n_permutations)
    ])
    return observed_t, float(np.mean(np.abs(null_t) >= np.abs(observed_t)))


def group_level_test(live_df: pd.DataFrame,
                     studio_df: pd.DataFrame) -> pd.DataFrame:
    """
    Run group-level permutation test for every ROI × hemisphere combination.
    """
    # Columns that are ROI accuracies (exclude metadata columns)
    meta_cols = {"participant", "group"}
    roi_cols  = [c for c in live_df.columns if c not in meta_cols]

    results = []
    print(f"\n{'='*60}")
    print(f"GROUP-LEVEL SURFACE MVPA RESULTS (permutation, n={N_PERMUTATIONS})")
    print(f"{'='*60}")

    for col in roi_cols:
        # Drop NaNs (parcels absent in one hemisphere)
        live_acc   = live_df[col].dropna().values
        studio_acc = studio_df[col].dropna().values

        if len(live_acc) < 3 or len(studio_acc) < 3:
            continue

        t_live,   p_live   = stats.ttest_1samp(live_acc,   0.5)
        t_studio, p_studio = stats.ttest_1samp(studio_acc, 0.5)
        t_btw, p_btw       = permutation_ttest(live_acc, studio_acc, N_PERMUTATIONS)

        pooled_sd = np.sqrt(
            (np.std(live_acc, ddof=1)**2 + np.std(studio_acc, ddof=1)**2) / 2
        )
        cohens_d = (np.mean(live_acc) - np.mean(studio_acc)) / pooled_sd \
                   if pooled_sd > 0 else np.nan

        sig = ("***" if p_btw < 0.001 else "**" if p_btw < 0.01
               else "*"  if p_btw < 0.05  else "ns")

        print(
            f"  {col:<35}  live={np.mean(live_acc):.3f}  "
            f"studio={np.mean(studio_acc):.3f}  "
            f"t={t_btw:+.3f}  p={p_btw:.4f} {sig}  d={cohens_d:+.3f}"
        )

        results.append({
            "ROI_hemi":           col,
            "live_mean_acc":      np.mean(live_acc),
            "live_sd":            np.std(live_acc, ddof=1),
            "live_vs_chance_p":   p_live,
            "studio_mean_acc":    np.mean(studio_acc),
            "studio_sd":          np.std(studio_acc, ddof=1),
            "studio_vs_chance_p": p_studio,
            "between_group_t":    t_btw,
            "between_group_p":    p_btw,
            "cohens_d":           cohens_d,
        })

    results_df = pd.DataFrame(results)

    from statsmodels.stats.multitest import multipletests
    _, p_fdr, _, _ = multipletests(results_df["between_group_p"], method="fdr_bh")
    results_df["between_group_p_fdr"] = p_fdr

    return results_df

In [53]:
# =============================================================================
# 10. MAIN
# =============================================================================

if __name__ == "__main__":

    # print("Loading Destrieux atlas (fsaverage5)...")
    atlas = load_destrieux_atlas()

    # Optionally print available label names for reference
    # print("\nAvailable Destrieux labels (left hemisphere):")
    # for lbl in atlas["lh"]["labels"]:
    #     print(f"  {lbl}")

    live_acc_df   = run_group_mvpa(LIVE_PARTICIPANTS,   shared_punchlines_live,
                                   1.5,   "live",   atlas)
    studio_acc_df = run_group_mvpa(STUDIO_PARTICIPANTS, shared_punchlines_pni,
                                   1.5, "studio", atlas)

    live_acc_df.to_csv(OUTPUT_DIR   / "live_surface_mvpa_accuracies_45.csv",   index=False)
    studio_acc_df.to_csv(OUTPUT_DIR / "studio_surface_mvpa_accuracies_45.csv", index=False)

    results_df = group_level_test(live_acc_df, studio_acc_df)
    results_df.to_csv(OUTPUT_DIR / "group_surface_mvpa_results_45.csv", index=False)
    print(f"\nAll results saved to {OUTPUT_DIR}/")

[fetch_surf_fsaverage] Dataset found in /scratch/network/vm5631/thesis/fsaverage6


Group: live  |  45 participants  |  TR=1.5s
Punchline events: 22

  [1/45] sub-025_task-pieman_space-fsaverage6_hemi-L_desc-clean.func.gii
    [lh] LSS — punchline trials...

  [3/45] sub-030_task-pieman_space-fsaverage6_hemi-L_desc-clean.func.gii
    [lh] LSS — punchline trials...

  [2/45] sub-044_task-pieman_space-fsaverage6_hemi-L_desc-clean.func.gii
    [lh] LSS — punchline trials...


    [lh] LSS — baseline trials...


    lh baseline:   0%|          | 0/22 [00:00<?, ?it/s]

    [lh] LSS — baseline trials...


...

    rh baseline:   0%|          | 0/22 [00:00<?, ?it/s] 2.78it/s]

    [rh] LSS — baseline trials...


    rh baseline:   0%|          | 0/22 [00:00<?, ?it/s]          

    [rh] LSS — baseline trials...



GROUP-LEVEL SURFACE MVPA RESULTS (permutation, n=5000)
  Heschl_lh                            live=0.712  studio=0.711  t=+0.047  p=0.9580 ns  d=+0.010
  STG_lateral_lh                       live=0.810  studio=0.798  t=+0.662  p=0.5092 ns  d=+0.139
  planum_temporale_lh                  live=0.742  studio=0.744  t=-0.121  p=0.9130 ns  d=-0.025
  STS_lh                               live=0.824  studio=0.819  t=+0.337  p=0.7526 ns  d=+0.071
  IFG_triangularis_lh                  live=0.755  studio=0.776  t=-1.150  p=0.2646 ns  d=-0.242
  IFG_opercularis_lh                   live=0.772  studio=0.777  t=-0.293  p=0.7652 ns  d=-0.062
  MTG_lh                               live=0.804  studio=0.814  t=-0.596  p=0.5416 ns  d=-0.126
  angular_gyrus_lh                     live=0.763  studio=0.770  t=-0.423  p=0.6654 ns  d=-0.089
  supramarginal_lh                     live=0.807  studio=0.795  t=+0.591  p=0.5740 ns  d=+0.125
  ACC_anterior_lh                      live=0.766  studio=0.794  t=-1.5